# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook guides you through loading, overviewing, and analyzing the FAIR² dataset using the `mlcroissant` library. You will learn how to extract tabular clinical, hormone, and genetic data by referencing entities strictly via their `@id` fields as defined by the Croissant schema.

### Dataset Source
The dataset schema is provided via this Croissant JSON-LD URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`. Fully reference all entities by their `@id` as per FAIR Data best practices.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata as object (not dict)
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their unique Croissant `@id`s. This ensures reproducibility and proper referencing in subsequent processing.

For each record set, list the record set `@id`, field `@id`s, and column `@id`s (if applicable).

In [ ]:
# List all record sets and their fields and columns by @id
record_sets = dataset.record_sets.keys()
print('Available RecordSets:')
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"\nRecordSet @id: {rs_id}")
    print(f"  Name: {getattr(rs, 'name', '<no name>')}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field_id in rs.fields.keys():
            field = rs.fields[field_id]
            print(f"    Field @id: {field_id}")
            print(f"      Name: {getattr(field, 'name', '<no name>')}")
            print(f"      DataType: {getattr(field, 'data_type', '<unknown>')}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col_id in rs.columns.keys():
            col = rs.columns[col_id]
            print(f"    Column @id: {col_id}")
            print(f"      Name: {getattr(col, 'name', '<no name>')}")
            print(f"      DataType: {getattr(col, 'data_type', '<unknown>')}")

## 3. Data Extraction
Extract data from each record set by referencing their `@id`s. Data is loaded into pandas DataFrames using record set `@id` and field/column `@id`.

Below, you can adjust the list of record set `@id`s for extraction as appropriate for your analysis.

In [ ]:
# Extract all tabular record sets into DataFrames
from collections import OrderedDict

# Gather all record set @id's
record_set_ids = list(dataset.record_sets.keys())
print(f"RecordSet @ids for extraction: {record_set_ids}")

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded RecordSet: {rs_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"\nUnable to load records from {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Choose a record set and numeric field (both referenced by `@id`) for basic analysis. Apply common EDA steps: filter, normalize numeric fields, and group by categories, always referencing by Croissant `@id`.

Example below: filtering and normalizing `age` for patients, grouped by `infertility`.

In [ ]:
# Choose a record set and field for analysis - adjust these according to dataset exploration above.
# Here, we demonstrate on the first available record set.

if record_set_ids:
    rs_id = record_set_ids[0]  # First record set
    df = dataframes[rs_id]

    # Find possible numeric fields by data type
    numeric_field_ids = []
    rs_fields = dataset.record_sets[rs_id].fields if hasattr(dataset.record_sets[rs_id], 'fields') else {}
    for field_id, field in rs_fields.items():
        dt = getattr(field, 'data_type', '')
        if dt and (dt.lower() in ['integer', 'float', 'number'] or 'integer' in dt.lower() or 'float' in dt.lower()):
            numeric_field_ids.append(field_id)
    # Default to first numeric field
    if numeric_field_ids:
        numeric_field = numeric_field_ids[0]
        print(f"Numeric field selected for EDA: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=['number']).columns[0] if not df.empty else None
        print(f"No numeric @id found in schema, using DataFrame numeric column: {numeric_field}")

    # Filtering: Example threshold
    threshold = 10
    if numeric_field and numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by categorical field
        group_field_candidates = [col for col in df.columns if col != numeric_field]
        group_field = None
        for gf in group_field_candidates:
            # Choose first bool/object dtype
            if str(df[gf].dtype) == 'bool' or str(df[gf].dtype) == 'object':
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (@id):")
            print(grouped_df)

## 5. Visualization
Visualize the numeric field's distribution and its relationship with the group field. All variables referenced by `@id`.

Below is an example using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of filtered numeric field, grouped by categorical
if record_set_ids and numeric_field and group_field and numeric_field in filtered_df and group_field in filtered_df:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (@id) after filtering")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} (@id) by {group_field} (@id)")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, you loaded and explored the FAIR² clinical-genetic dataset using `mlcroissant` referencing all entities by their Croissant `@id`. You extracted and visualized core patient-level tabulations and learned how to filter, normalize, and group by medically significant attributes in reproducible FAIR style.

Next steps could involve more detailed analysis, linkage between record sets, or custom queries on genetic variants. For any downstream use, always cite the dataset and adhere to the licensing and data sensitivity policies described in the metadata.